# Task 5: Mental Health Support Chatbot (Fine-Tuning Workflow & API Fallback)
**Internship Task - DevelopersHub Corporation**

### Objective
Build a basic chatbot that provides supportive and empathetic responses for stress, anxiety, and emotional wellness. 

### Notebook Structure
1. **Fine-Tuning Workflow**: Full Python code utilizing Hugging Face's `transformers` and `datasets` to fine-tune `DistilGPT2` on the `EmpatheticDialogues` dataset (designed to be run in a GPU-accelerated environment like Google Colab).
2. **Empathetic API Fallback**: A lightweight runtime using **Gemini 2.5 Flash Lite** configured with a custom empathetic system instruction, allowing you to run and evaluate the conversational agent directly on your CPU/laptop.

--- 
# Part 1: Fine-Tuning Workflow (Google Colab / GPU Ready)
This section outlines the exact steps to download the `empathetic_dialogues` dataset, tokenize it, and train `DistilGPT2` using Hugging Face's Trainer API.

In [ ]:
# Install required Hugging Face libraries (Run this in Google Colab)
# !pip install transformers datasets accelerate torch

In [ ]:
# Preprocessing and training code outline
"""
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

# 1. Load the EmpatheticDialogues dataset
dataset = load_dataset("empathetic_dialogues", split="train[:5000]") # Load subset for faster training

# 2. Load pre-trained Tokenizer and Model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Tokenize Dataset
def tokenize_function(examples):
    # Combine context and response
    texts = [f"User: {c} \nAssistant: {r}" for c, r in zip(examples['prompt'], examples['utterance'])]
    inputs = tokenizer(texts, truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./empathetic-chatbot",
    evaluation_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_steps=500,
    fp16=torch.cuda.is_available(),
    logging_steps=100
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# 6. Start Fine-Tuning
# trainer.train()
"""
print("Workflow code template loaded!")

--- 
# Part 2: Empathetic Chatbot Runtime (Gemini API / Offline Fallback)
Below is the lightweight script to run the chatbot using Gemini 2.5 Flash Lite or local simulation.

In [ ]:
# !pip install google-generativeai

In [ ]:
import os
import getpass
import google.generativeai as genai
print("Gemini SDK imported successfully!")

In [ ]:
# Configure key and detect mode
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("LLM_API_KEY")
if not api_key:
    api_key = getpass.getpass("Please enter your Gemini API key (or press Enter to run offline): ").strip()

if api_key.startswith("AIzaSy"):
    client_type = "gemini"
    genai.configure(api_key=api_key)
    print("Configured Gemini API Client.")
else:
    client_type = "offline"
    print("No valid Gemini Key. Running chatbot in local offline simulation mode.")

## Step 3: Define Empathetic Persona
The system instructions are engineered to instruct the LLM to act as a gentle, emotionally supportive companion for stress and anxiety.

In [ ]:
EMPATHETIC_SYSTEM_PROMPT = (
    "You are a gentle, kind, and empathetic mental health support chatbot. "
    "Your goal is to provide supportive, warm, and validation-focused responses to help users "
    "deal with stress, anxiety, or emotional wellness. You active-listen, normalize their emotions, "
    "and offer gentle, non-judgmental guidance. "
    "Always end your response with: 'Note: I am here to support you, but I am an AI companion, not a licensed therapist. If you are experiencing a crisis, please seek professional support.'"
)
print("Empathetic System Prompt defined!")

In [ ]:
def get_offline_empathetic_response(query):
    """
    Mock response dictionary representing high-quality empathetic responses.
    """
    query_lower = query.lower()
    disclaimer = "\n\nNote: I am here to support you, but I am an AI companion, not a licensed therapist. If you are experiencing a crisis, please seek professional support."
    
    if "stressed" in query_lower or "stress" in query_lower:
        return (
            "I hear you, and I'm really sorry you're feeling so overwhelmed right now. Stress can feel like carrying "
            "an incredibly heavy weight. Please remember to take a deep breath. You don't have to figure everything "
            "out this very second. Can you try taking three slow, deep breaths with me? I am here to listen if you want "
            "to vent about what's going on."
            + disclaimer
        )
    elif "anxious" in query_lower or "anxiety" in query_lower:
        return (
            "It is completely understandable to feel anxious, and I want you to know your feelings are valid. "
            "Anxiety can make our thoughts race and make us feel unsafe. Let's try to ground ourselves. Can you name "
            "three things in the room around you that you can see? Focusing on the physical space can sometimes help quiet "
            "the noise. You are safe here."
            + disclaimer
        )
    else:
        return (
            "Thank you for sharing that with me. It takes courage to open up about how we feel. I want to assure you "
            "that I am here to offer a safe, warm, and supportive space for you. Your feelings are important, and you "
            "don't have to go through this alone."
            + disclaimer
        )

In [ ]:
def ask_empathetic_chatbot(query):
    if client_type == "offline":
        return get_offline_empathetic_response(query)
    
    try:
        model = genai.GenerativeModel(
            model_name='gemini-2.5-flash-lite',
            system_instruction=EMPATHETIC_SYSTEM_PROMPT
        )
        response = model.generate_content(query)
        return response.text
    except Exception as e:
        print(f"[Gemini API Failed: {str(e)}]. Falling back to offline simulation...")
        return get_offline_empathetic_response(query)

## Step 4: Test Chatbot Responses

In [ ]:
# Test Query 1
q1 = "I have been feeling extremely stressed about my exams lately."
print(f"User: {q1}\n")
print(f"Chatbot:\n{ask_empathetic_chatbot(q1)}")

In [ ]:
# Test Query 2
q2 = "I feel anxious and my heart is beating fast."
print(f"User: {q2}\n")
print(f"Chatbot:\n{ask_empathetic_chatbot(q2)}")